# Inquiry Arena: EntailmentBank scenarios and batch runs

Run the cells from top to bottom (click a cell, then the ▶ button, or press Shift+Enter).
Everything runs on Google's computers; nothing is installed on yours.

You need three files from the chat: `app.py` (v0.5), `convert_entailmentbank.py` and `batch_run.py`.

## 1. Upload the three files
A file picker opens: select all three at once.

In [ ]:
from google.colab import files
uploaded = files.upload()
print(sorted(uploaded))

## 2. Install what the scripts need (about a minute)

In [ ]:
!pip -q install streamlit openai requests datasets

## 3. Download EntailmentBank (it is on GitHub)

In [ ]:
!git clone -q --depth 1 https://github.com/allenai/entailment_bank
EB = "entailment_bank/data/public_dataset/entailment_trees_emnlp2021_data_v2/dataset/task_2"
!ls -la {EB}

## 4. Make the scenarios
This makes one file per EntailmentBank item. The answer options come from ARC, which is downloaded automatically (`--arc hf`).

Settings you may want to change:
- `--split`: `dev` (187 items) or `test` (340) for results; `train` (1,313) for trying things out
- `--agents`: 2 or 3
- `--mode`: `step` (premises of each proof step go to different agents), `subtree` (each agent gets whole branches) or `random`
- `--distractors`: distractors per agent on top of its proof facts (a number, or `all`)
- `--seed`: change it to get a different split of the same items

In [ ]:
!python convert_entailmentbank.py --eb-dir {EB} --arc hf --split dev --agents 2 --mode step --seed 1 --distractors 3 --out scenarios/dev_n2_step_s1

## 5. Look at one scenario (and download it to try in the app's uploader)

In [ ]:
import json, glob
fn = sorted(glob.glob("scenarios/dev_n2_step_s1/scenario_*.json"))[0]
sc = json.load(open(fn))
print(sc["title"]); print(sc["question"]); print(sc["options"], "answer:", sc["answer"])
for a in sc["agents"]:
    print(f"\n{a['name']}:")
    for k, v in a["facts"].items():
        print(f"  {k} [{sc['fact_types'][k]}] {v}")
print("\nGold tree:"); print("\n".join(sc["gold_tree"]))
files.download(fn)

## 6. Test run with fake players (free, takes seconds)

In [ ]:
!python batch_run.py --app app.py --scenarios scenarios/dev_n2_step_s1 --out results/mock --mock --limit 3

## 7. Add your OpenRouter key
Click the key icon in the left sidebar, add a secret named `OPENROUTER_API_KEY` with your key as the value,
and switch on "Notebook access". Then run this cell. The key is never printed or saved in the notebook.

In [ ]:
from google.colab import userdata
import os
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
print("key loaded")

## 8. Pilot with real models
5 scenarios × 4 strategies = 20 dialogues. `--max-cost` stops starting new dialogues once the estimated cost passes that many dollars.
If the run stops or the page disconnects, run the cell again: finished dialogues are skipped.

In [ ]:
!python batch_run.py --app app.py --scenarios scenarios/dev_n2_step_s1 --out results/pilot \
    --strategies free goals assumptions contrast --models openai/gpt-4o-mini --judge openai/gpt-4o-mini \
    --limit 5 --max-cost 1

## 9. Look at the results

In [ ]:
import pandas as pd
df = pd.read_csv("results/pilot/summary.csv")
print(df.groupby("strategy")[["judge_correct", "justified_proxy", "pooled_correct", "question_only_correct",
                              "private_facts_shared", "turns", "cost_usd"]].mean().round(2))
df.head()

## 10. Save everything
Colab deletes files when the session ends, so download a zip (or copy it to your Google Drive).

In [ ]:
!zip -q -r inquiry_arena_results.zip scenarios results
files.download("inquiry_arena_results.zip")